In [1]:
# ============================================
# BLOOD DONOR ELIGIBILITY - RANDOM FOREST (TUNED + CV)
# ============================================

from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split

# --------------------------------------------
# 1. Load Dataset (local first, fallback to DATASET folder)
# --------------------------------------------

candidate_paths = [
    Path("fixed_blood_donor_dataset.csv"),
    Path("../DATASET/Eligibility model/blood_donation.csv"),
]

dataset_path = next((p for p in candidate_paths if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Could not find fixed_blood_donor_dataset.csv")

df = pd.read_csv(dataset_path)
print(f"Loaded: {dataset_path}")
print("Dataset Shape:", df.shape)

# --------------------------------------------
# 2. Define Features and Target
# --------------------------------------------

target_col = "Eligible_for_Donation_binary"
X = df.drop(columns=[target_col])
y = df[target_col]

print("Target distribution (%):")
print((y.value_counts(normalize=True) * 100).round(2))

# --------------------------------------------
# 3. Stratified Train-Test Split
# --------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

# --------------------------------------------
# 4. Hyperparameter Search with Regularization Controls
# --------------------------------------------

base_rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
)

param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [6, 10, 14],
    "min_samples_split": [5, 10, 20],
    "min_samples_leaf": [2, 4, 8],
    "max_features": ["sqrt", 0.5],
    "class_weight": ["balanced", "balanced_subsample"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    estimator=base_rf,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

search.fit(X_train, y_train)
best_rf = search.best_estimator_

print("\nBest Parameters:")
print(search.best_params_)
print(f"Best CV F1: {search.best_score_:.4f}")

# --------------------------------------------
# 5. Evaluate Train vs Test (Overfitting Check)
# --------------------------------------------

train_pred = best_rf.predict(X_train)
test_pred = best_rf.predict(X_test)
train_proba = best_rf.predict_proba(X_train)[:, 1]
test_proba = best_rf.predict_proba(X_test)[:, 1]

train_f1 = f1_score(y_train, train_pred)
test_f1 = f1_score(y_test, test_pred)
train_auc = roc_auc_score(y_train, train_proba)
test_auc = roc_auc_score(y_test, test_proba)

overfit_gap = train_f1 - test_f1

print(f"\nTrain Accuracy: {accuracy_score(y_train, train_pred):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, test_pred):.4f}")
print(f"Train F1:       {train_f1:.4f}")
print(f"Test F1:        {test_f1:.4f}")
print(f"Train ROC-AUC:  {train_auc:.4f}")
print(f"Test ROC-AUC:   {test_auc:.4f}")
print(f"Overfitting Gap (Train F1 - Test F1): {overfit_gap:.4f}")

if overfit_gap > 0.05:
    print("Warning: Noticeable overfitting detected. Tighten depth/leaf constraints.")
else:
    print("Good: Overfitting gap is acceptable.")

print("\nConfusion Matrix (Test):")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report (Test):")
print(classification_report(y_test, test_pred, digits=4))

# --------------------------------------------
# 6. Feature Importance
# --------------------------------------------

feature_importance = (
    pd.Series(best_rf.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
)

print("\nFeature Importance:")
print(feature_importance)

Loaded: fixed_blood_donor_dataset.csv
Dataset Shape: (10000, 9)
Target distribution (%):
Eligible_for_Donation_binary
1    64.16
0    35.84
Name: proportion, dtype: float64
Training size: (8000, 8)
Testing size: (2000, 8)
Fitting 5 folds for each of 216 candidates, totalling 1080 fits

Best Parameters:
{'class_weight': 'balanced', 'max_depth': 6, 'max_features': 0.5, 'min_samples_leaf': 8, 'min_samples_split': 5, 'n_estimators': 200}
Best CV F1: 0.8837

Train Accuracy: 0.8313
Test Accuracy:  0.8320
Train F1:       0.8838
Test F1:        0.8842
Train ROC-AUC:  0.8345
Test ROC-AUC:   0.7659
Overfitting Gap (Train F1 - Test F1): -0.0004
Good: Overfitting gap is acceptable.

Confusion Matrix (Test):
[[ 381  336]
 [   0 1283]]

Classification Report (Test):
              precision    recall  f1-score   support

           0     1.0000    0.5314    0.6940       717
           1     0.7925    1.0000    0.8842      1283

    accuracy                         0.8320      2000
   macro avg     0.